# M.M.M Make Mincraft Mode

`모두 실행`을 기준으로 실행합니다. 중간 승인 입력 없이 자동으로 진행합니다.

- **Full**: 새 플랜 생성 → 자동 저장 → 제작
- **Plan**: 새 플랜 생성 → 자동 저장 → 종료
- **Revise**: 기존 source/release ZIP 업로드 → 수정 플랜 생성 → 자동 저장 → 기존 모드 수정 제작
- **Execute**: 저장된 플랜 로드 → 바로 제작
- **Debug**: 모델 생성 없이 프로젝트 전체 진단을 끝까지 실행하고 실패 항목/로그/JSON을 수집

1번 설정 셀에서 **Minecraft 버전은 드롭다운으로 선택**하고, **loader**, **참고할 GitHub 모드 repository URL**을 설정할 수 있습니다. 버전/loader를 `Auto` 또는 빈 값으로 두면 기존 host-owned platform selector가 결정하며, Revise에서 Auto이면 기존 프로젝트 target을 보존합니다. 참고 링크는 복수 입력할 수 있고 실제 재사용은 commit pin, 라이선스, target compatibility, dependency closure, compile/proof 검증을 통과한 source만 허용됩니다.

Qwen 계열 모델은 각 단계에서 허용된 MMM/MCP 툴을 자동으로 선택하고, 툴 결과를 다시 받아 필요한 만큼 이어서 호출합니다. 사람 승인 대신 proposal content hash 검증은 내부에서 자동으로 유지됩니다.

원격 trajectory/temporary-skill 저장은 기본적으로 꺼져 있으며, 1번 설정 셀에서 사용자가 명시적으로 동의한 경우에만 활성화됩니다.


In [ ]:
# @title 1. 실행 모드 및 설정
import os

RUN_MODE = "Full" #@param ["Full", "Plan", "Revise", "Execute", "Debug"]
PROMPT = "계절마다 다른 작물을 재배하고 요리하는 모드를 만들어줘." #@param {type:"string"}
PLAN_FILE = "" #@param {type:"string"}
MINECRAFT_VERSION = "Auto" #@param ["Auto", "26.2", "26.1.1", "26.1", "1.21.10", "1.21.8", "1.21.5", "1.21.4", "1.21.1", "1.20.6", "1.20.4", "1.20.1"]
MOD_LOADER = "Auto" #@param {type:"string"}
REFERENCE_MOD_URLS = "" #@param {type:"string"}
MODEL_PROFILE = "Qwen3.5-9B_6GB" #@param ["Qwen3.5-9B_6GB", "Qwen3.6-35B_23GB", "Qwen3.8-27B_18GB", "fast_test", "remote_quality"]
PERFORMANCE_MODE = "Auto" #@param ["Auto", "Latency", "Throughput"]
KV_CACHE_QUANT = "q4_0" #@param ["q4_0", "q8_0", "f16"]
KV_CACHE_AUTOTUNE = True #@param {type:"boolean"}
FAST_MODE = False #@param {type:"boolean"}
SAVE_TO_GOOGLE_DRIVE = True #@param {type:"boolean"}
ALLOW_REMOTE_TRAJECTORY_STORE = False #@param {type:"boolean"}
CURSEFORGE_API_KEY = "" #@param {type:"string"}

REMOTE_BASE_URL, REMOTE_TEXT_MODEL, REMOTE_IMAGE_MODEL, REMOTE_SPEECH_MODEL, RUN_BLOCKBENCH, RUN_RUNTIME, RUN_CLIENT, RUN_MINEFLAYER, RUN_VISUAL_REVIEW, ACCEPT_EULA, SERVER_LAUNCHER, RUN_NAME, SCREENSHOTS = "", "", "", "", False, False, False, False, False, False, "", "complete-colab-run", []

performance_mode = str(PERFORMANCE_MODE).strip().lower()
if performance_mode not in {"auto", "latency", "throughput"}:
    raise ValueError("PERFORMANCE_MODE은 Auto, Latency, Throughput 중 하나여야 합니다.")
os.environ["MMM_PERFORMANCE_MODE"] = performance_mode
os.environ.setdefault("MMM_PLATFORM_DISCOVERY_RETRIES", "4")

reference_mod_urls = str(REFERENCE_MOD_URLS or "").strip()
if reference_mod_urls:
    os.environ["MMM_REFERENCE_MOD_URLS"] = reference_mod_urls
else:
    os.environ.pop("MMM_REFERENCE_MOD_URLS", None)

curseforge_api_key = str(CURSEFORGE_API_KEY or "").strip()
if not curseforge_api_key:
    try:
        from google.colab import userdata as colab_userdata
        curseforge_api_key = str(colab_userdata.get("CURSEFORGE_API_KEY") or "").strip()
    except Exception:
        curseforge_api_key = ""
if curseforge_api_key:
    os.environ["MMM_CURSEFORGE_API_KEY"] = curseforge_api_key
else:
    os.environ.pop("MMM_CURSEFORGE_API_KEY", None)

VALID_RUN_MODES = {"Full", "Plan", "Revise", "Execute", "Debug"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"지원하지 않는 실행 모드: {RUN_MODE}")
if RUN_MODE not in {"Execute", "Debug"} and not PROMPT.strip():
    raise ValueError("선택한 실행 모드에서는 PROMPT를 입력해야 합니다.")
if RUN_RUNTIME and not ACCEPT_EULA:
    raise ValueError("Minecraft 실행 검증에는 EULA 동의가 필요합니다.")


In [ ]:
# @title 2. GitHub 최신 main 설치
import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

os.environ["MMM_REMOTE_TRAJECTORY_STORE_CONSENT"] = "1" if ALLOW_REMOTE_TRAJECTORY_STORE else "0"
REPO_DIR = Path("/content/M.M.M-Make-Mincraft-Mode")
EXPECTED_REPOSITORY = "https://github.com/jujumelona/M.M.M-Make-Mincraft-Mode.git"

def sync_main_checkout():
    previous = ""
    if (REPO_DIR / ".git").is_dir():
        origin_url = subprocess.check_output(["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"], text=True).strip()
        normalized_origin = origin_url.rstrip("/").removesuffix(".git")
        normalized_expected = EXPECTED_REPOSITORY.removesuffix(".git")
        if normalized_origin != normalized_expected:
            raise RuntimeError("Existing Colab checkout is not the official M.M.M GitHub repository. Remove that checkout and rerun setup cell 2.")
        previous = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "+main:refs/remotes/origin/main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-f", "main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "refs/remotes/origin/main"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "clean", "-fd"], check=True)
    elif REPO_DIR.exists():
        raise RuntimeError(f"Git 저장소가 아닌 경로가 이미 있습니다: {REPO_DIR}")
    else:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "main", EXPECTED_REPOSITORY, str(REPO_DIR)], check=True)
    return previous

def current_main_commit():
    return subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

def assert_clean_current_main():
    used = current_main_commit()
    remote = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "refs/remotes/origin/main"], text=True).strip()
    if used != remote:
        raise RuntimeError("The checkout is not exactly GitHub origin/main. Rerun setup cell 2.")
    tracked = subprocess.check_output(["git", "-C", str(REPO_DIR), "status", "--porcelain", "--untracked-files=no"], text=True).strip()
    if tracked:
        raise RuntimeError("The Colab engine checkout contains tracked local changes. Rerun setup cell 2.")
    return used

transformers_was_loaded = "transformers" in sys.modules
engine_was_loaded = any(name == "minecraft_mod_ai" or name.startswith("minecraft_mod_ai.") for name in sys.modules)
loaded_engine_module = sys.modules.get("minecraft_mod_ai")
engine_module_file = getattr(loaded_engine_module, "__file__", "") or ""
previous_commit = sync_main_checkout()
USED_COMMIT = assert_clean_current_main()
REMOTE_COMMIT = USED_COMMIT
print("GitHub commit:", USED_COMMIT, flush=True)

if RUN_MODE == "Debug":
    print("Debug setup: generation model/runtime setup is skipped.", flush=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[dev]", "build"], check=True)
    OUTPUT_ROOT = str(REPO_DIR / "audit")
    SETUP_STATE = SETUP_RECEIPT = SETUP_FINGERPRINT = COLAB_SETUP_MODULE = None
else:
    setup_script = REPO_DIR / "tools" / "colab_runtime_setup.py"
    if not setup_script.is_file():
        raise FileNotFoundError(f"Pulled commit has no Colab setup script: {setup_script}")
    setup_module_name = f"_mmm_colab_runtime_setup_{USED_COMMIT[:12]}"
    setup_spec = importlib.util.spec_from_file_location(setup_module_name, setup_script)
    if setup_spec is None or setup_spec.loader is None:
        raise RuntimeError(f"Cannot load Colab setup script: {setup_script}")
    COLAB_SETUP_MODULE = importlib.util.module_from_spec(setup_spec)
    sys.modules[setup_module_name] = COLAB_SETUP_MODULE
    setup_spec.loader.exec_module(COLAB_SETUP_MODULE)
    SETUP_STATE = COLAB_SETUP_MODULE.setup_colab_runtime(
        repo_dir=REPO_DIR, used_commit=USED_COMMIT, model_profile=MODEL_PROFILE,
        save_to_google_drive=SAVE_TO_GOOGLE_DRIVE, remote_base_url=REMOTE_BASE_URL,
        remote_text_model=REMOTE_TEXT_MODEL, remote_image_model=REMOTE_IMAGE_MODEL,
        remote_speech_model=REMOTE_SPEECH_MODEL, transformers_was_loaded=transformers_was_loaded,
        engine_was_loaded=engine_was_loaded, engine_module_file=engine_module_file, previous_commit=previous_commit,
    )
    REPO_DIR = Path(SETUP_STATE["repo_dir"])
    OUTPUT_ROOT = SETUP_STATE["output_root"]
    SETUP_RECEIPT = SETUP_STATE["receipt"]
    SETUP_FINGERPRINT = SETUP_STATE["setup_fingerprint"]


In [ ]:
# @title 3. 기존 모드 입력
import os
import subprocess

os.environ["MMM_REMOTE_TRAJECTORY_STORE_CONSENT"] = "1" if ALLOW_REMOTE_TRAJECTORY_STORE else "0"
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "+main:refs/remotes/origin/main"], check=True)
LATEST_MAIN_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "refs/remotes/origin/main"], text=True).strip()
CURRENT_ENGINE_COMMIT = current_main_commit()
if CURRENT_ENGINE_COMMIT != LATEST_MAIN_COMMIT:
    print("GitHub main changed after setup; refreshing engine:", CURRENT_ENGINE_COMMIT[:12], "->", LATEST_MAIN_COMMIT[:12], flush=True)
    previous_commit = sync_main_checkout()
    USED_COMMIT = assert_clean_current_main()
    REMOTE_COMMIT = USED_COMMIT
    if RUN_MODE != "Debug":
        setup_script = REPO_DIR / "tools" / "colab_runtime_setup.py"
        setup_module_name = f"_mmm_colab_runtime_setup_{USED_COMMIT[:12]}"
        setup_spec = importlib.util.spec_from_file_location(setup_module_name, setup_script)
        if setup_spec is None or setup_spec.loader is None:
            raise RuntimeError(f"Cannot reload Colab setup script: {setup_script}")
        COLAB_SETUP_MODULE = importlib.util.module_from_spec(setup_spec)
        sys.modules[setup_module_name] = COLAB_SETUP_MODULE
        setup_spec.loader.exec_module(COLAB_SETUP_MODULE)
        SETUP_STATE = COLAB_SETUP_MODULE.setup_colab_runtime(
            repo_dir=REPO_DIR, used_commit=USED_COMMIT, model_profile=MODEL_PROFILE,
            save_to_google_drive=SAVE_TO_GOOGLE_DRIVE, remote_base_url=REMOTE_BASE_URL,
            remote_text_model=REMOTE_TEXT_MODEL, remote_image_model=REMOTE_IMAGE_MODEL,
            remote_speech_model=REMOTE_SPEECH_MODEL, transformers_was_loaded="transformers" in sys.modules,
            engine_was_loaded=True, engine_module_file=getattr(sys.modules.get("minecraft_mod_ai"), "__file__", "") or "", previous_commit=previous_commit,
        )
        REPO_DIR = Path(SETUP_STATE["repo_dir"])
        OUTPUT_ROOT = SETUP_STATE["output_root"]
        SETUP_RECEIPT = SETUP_STATE["receipt"]
        SETUP_FINGERPRINT = SETUP_STATE["setup_fingerprint"]

if RUN_MODE == "Debug":
    EXISTING_INPUT = None
    print("Debug: 기존 모드 입력 생략", flush=True)
else:
    from minecraft_mod_ai.colab_run_modes import prepare_existing_mod_input
    EXISTING_INPUT = prepare_existing_mod_input(RUN_MODE)


In [ ]:
# @title 4. 설치 확인
def assert_current_colab_setup():
    if RUN_MODE == "Debug":
        assert_clean_current_main()
        return
    COLAB_SETUP_MODULE.assert_setup_state(
        SETUP_STATE, repo_dir=REPO_DIR, used_commit=USED_COMMIT, model_profile=MODEL_PROFILE,
        save_to_google_drive=SAVE_TO_GOOGLE_DRIVE, remote_base_url=REMOTE_BASE_URL,
        remote_text_model=REMOTE_TEXT_MODEL, remote_image_model=REMOTE_IMAGE_MODEL, remote_speech_model=REMOTE_SPEECH_MODEL,
    )
    expected_consent = "1" if ALLOW_REMOTE_TRAJECTORY_STORE else "0"
    if os.environ.get("MMM_REMOTE_TRAJECTORY_STORE_CONSENT") != expected_consent:
        raise RuntimeError("원격 trajectory 저장 동의 상태가 설정 이후 변경되었습니다. 1번 셀에서 선택한 뒤 2번 셀부터 다시 실행하세요.")

assert_current_colab_setup()
if RUN_MODE == "Debug":
    print("설치 확인: Debug 진단 준비 완료")
else:
    from minecraft_mod_ai import ModelRegistry
    REGISTRY_PATH = REPO_DIR / "config/model_registry.yaml"
    if not REGISTRY_PATH.is_file():
        raise FileNotFoundError(REGISTRY_PATH)
    registry_manager = ModelRegistry()
    registry = registry_manager.to_public_dict()
    if MODEL_PROFILE not in registry["profiles"]:
        raise ValueError(f"지원하지 않는 모델 프로필: {MODEL_PROFILE}")
    planner_config = registry_manager.role(MODEL_PROFILE, "planner")
    print("모델 프로필:", MODEL_PROFILE)
    print("기획 모델:", planner_config.model_id)
    print("기획 백엔드:", planner_config.provider, "/", planner_config.adapter)
    print("기획 양자화:", planner_config.quantization or "none")
    print("기획 native context:", f"{planner_config.max_context:,} tokens")
    print("Minecraft target 요청:", MINECRAFT_VERSION or "Auto")
    print("Loader 요청:", MOD_LOADER or "Auto")
    print("참고 모드 repository:", REFERENCE_MOD_URLS or "없음")
    print("결과 저장 위치:", OUTPUT_ROOT)
    from minecraft_mod_ai.custom_module_generator import _extract_json
    sample_json = _extract_json('{"operations": [], "runtime_tests": [], "complete": true, "next_cursor": ""}')
    assert "operations" in sample_json, "Engine JSON parser self-check failed."
    print("설치 확인: 완료")


In [ ]:
# @title 5. 플랜 생성/불러오기 및 자동 진행
import logging
import os
import traceback
from pathlib import Path

assert_current_colab_setup()
PLAN_DIALOG = reply = FINAL_PLAN_PATH = PLAN_PATH = session = None
PLAN_ERROR = PLAN_ERROR_PATH = None
if RUN_MODE == "Debug":
    print("Debug: 플랜 생성과 모델 실행을 생략하고 전체 진단으로 진행합니다.", flush=True)
else:
    from minecraft_mod_ai import CompleteModAISession
    from minecraft_mod_ai.colab_run_modes import resolve_plan_path, run_plan_dialog
    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
    for _logger_name in ("minecraft_mod_ai.platform_live_discovery", "minecraft_mod_ai.platform_catalog", "minecraft_mod_ai.platform_optimizer", "minecraft_mod_ai.reuse_planner"):
        logging.getLogger(_logger_name).setLevel(logging.INFO)
    os.environ["MMM_LLAMA_KV_AUTOTUNE"] = "1" if KV_CACHE_AUTOTUNE else "0"
    os.environ["MMM_AGENT_TOOLS"] = "1"
    print("성능 모드:", performance_mode, "(실측 자동 튜닝)")
    try:
        session = CompleteModAISession(
            output_root=OUTPUT_ROOT,
            minecraft_version=MINECRAFT_VERSION,
            loader=MOD_LOADER,
            model_profile=MODEL_PROFILE,
            existing_input=EXISTING_INPUT,
            fast_mode=FAST_MODE,
            kv_cache_quant=KV_CACHE_QUANT,
        )
        PLAN_PATH = resolve_plan_path(run_mode=RUN_MODE, output_root=OUTPUT_ROOT, configured_path=PLAN_FILE)
        PLAN_DIALOG = run_plan_dialog(session=session, run_mode=RUN_MODE, prompt=PROMPT, plan_path=PLAN_PATH)
        reply = PLAN_DIALOG.reply
        FINAL_PLAN_PATH = PLAN_DIALOG.plan_path
        print("플랜 준비 완료:", FINAL_PLAN_PATH)
        print("Qwen/MCP tool calling: enabled")
    except Exception as exc:
        PLAN_ERROR = exc
        PLAN_ERROR_PATH = Path(OUTPUT_ROOT) / "planner_failure.log"
        PLAN_ERROR_PATH.parent.mkdir(parents=True, exist_ok=True)
        PLAN_ERROR_PATH.write_text(traceback.format_exc(), encoding="utf-8")
        print("플랜 단계가 완료되지 않았습니다. 제작은 보류하고 전체 원인 로그를 남겼습니다.", flush=True)
        print("플랜 오류:", repr(exc), flush=True)
        traceback.print_exc()
        print("플랜 오류 로그:", PLAN_ERROR_PATH, flush=True)


In [ ]:
# @title 6. 제작 또는 Debug 전체 진단
import json
import subprocess
import sys

assert_current_colab_setup()
BUILD_RESULT = None
DEBUG_REPORT_PATH = DEBUG_LOG_PATH = None
if RUN_MODE == "Debug":
    audit_script = REPO_DIR / "tools" / "full_project_audit.py"
    if not audit_script.is_file():
        raise FileNotFoundError(audit_script)
    print("Debug: 전체 프로젝트 진단 시작 — 한 검사가 실패해도 나머지 검사를 계속합니다.", flush=True)
    subprocess.run([sys.executable, str(audit_script)], cwd=REPO_DIR, check=False)
    DEBUG_REPORT_PATH = REPO_DIR / "audit" / "FULL_PROJECT_AUDIT.json"
    DEBUG_LOG_PATH = REPO_DIR / "audit" / "FULL_PROJECT_AUDIT.log"
    if not DEBUG_REPORT_PATH.is_file():
        raise FileNotFoundError(f"Debug report was not produced: {DEBUG_REPORT_PATH}")
    DEBUG_REPORT = json.loads(DEBUG_REPORT_PATH.read_text(encoding="utf-8"))
    print("Debug overall:", DEBUG_REPORT.get("overall_status"))
    print("실패 검사:", DEBUG_REPORT.get("failed_checks", []))
    for check in DEBUG_REPORT.get("checks", []):
        state = "PASS" if check.get("passed") else "FAIL"
        print(f"[{state}] {check.get('name')}: {check.get('detail')}")
    print("Debug JSON:", DEBUG_REPORT_PATH)
    print("Debug log:", DEBUG_LOG_PATH)
else:
    from minecraft_mod_ai import CompleteExecutionOptions
    from minecraft_mod_ai.colab_run_modes import should_build
    if not should_build(RUN_MODE):
        print("Plan: 제작 생략")
    elif PLAN_ERROR is not None:
        print("플랜 실패로 제작을 보류했습니다. 오류 로그:", PLAN_ERROR_PATH, flush=True)
    else:
        print("모드 생성: 시작", flush=True)
        options = CompleteExecutionOptions(run_blockbench=RUN_BLOCKBENCH, run_runtime=RUN_RUNTIME, run_client=RUN_CLIENT, run_mineflayer=RUN_MINEFLAYER, run_visual_review=RUN_VISUAL_REVIEW, eula_accepted=ACCEPT_EULA, server_launcher=SERVER_LAUNCHER or None, screenshot_paths=tuple(SCREENSHOTS), resume=True)
        BUILD_RESULT = session.build(reply, run_name=RUN_NAME, options=options)
        print("제작 상태:", BUILD_RESULT.status)
        print("프로젝트:", BUILD_RESULT.project_root)
        print("결과 ZIP:", BUILD_RESULT.release_zip)
        if BUILD_RESULT.run_resumed:
            print("재개 실행: 예")
        if BUILD_RESULT.quality_report:
            print("품질 검증:", BUILD_RESULT.quality_report["overall_status"])
        if BUILD_RESULT.unresolved_gates:
            print("미해결 항목:", ", ".join(BUILD_RESULT.unresolved_gates))


In [ ]:
# @title 7. 결과 다운로드
from pathlib import Path
import zipfile

if RUN_MODE == "Debug":
    debug_files = [Path(p) for p in (DEBUG_REPORT_PATH, DEBUG_LOG_PATH) if p is not None and Path(p).is_file()]
    if not debug_files:
        raise FileNotFoundError("다운로드할 Debug 결과가 없습니다.")
    debug_zip = Path("/content/mmm-debug-report.zip")
    with zipfile.ZipFile(debug_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in debug_files:
            archive.write(path, arcname=path.name)
    print("Debug 결과 ZIP:", debug_zip)
    try:
        from google.colab import files as colab_files
        colab_files.download(str(debug_zip))
    except ImportError:
        print("로컬 경로:", debug_zip.resolve())
elif RUN_MODE == "Plan" and FINAL_PLAN_PATH is None:
    print("플랜 생성 실패로 다운로드할 파일이 없습니다. 오류 로그:", PLAN_ERROR_PATH)
elif RUN_MODE == "Plan":
    plan_path = Path(FINAL_PLAN_PATH)
    if not plan_path.is_file():
        raise FileNotFoundError(plan_path)
    print("플랜 파일:", plan_path)
    try:
        from google.colab import files as colab_files
        colab_files.download(str(plan_path))
    except ImportError:
        print("로컬 경로:", plan_path.resolve())
elif BUILD_RESULT is None or not BUILD_RESULT.release_zip:
    print("다운로드할 제작 결과가 없습니다.")
else:
    release_zip = Path(BUILD_RESULT.release_zip)
    if not release_zip.is_file():
        raise FileNotFoundError(release_zip)
    print("release:", release_zip)
    print("size:", release_zip.stat().st_size, "bytes")
    try:
        from google.colab import files as colab_files
        colab_files.download(str(release_zip))
    except ImportError:
        print("로컬 경로:", release_zip.resolve())


## 실행 모드

기본값은 **Full**입니다.

**Full**은 플랜을 생성·저장한 뒤 제작으로 바로 이어집니다. **Plan**은 플랜만 저장합니다. **Revise**는 기존 source/release ZIP을 수정 제작하고, **Execute**는 저장된 proposal JSON을 바로 제작합니다.

`MINECRAFT_VERSION`은 1번 설정 셀의 드롭다운에서 선택합니다. `Auto`이면 host selector에 결정을 맡깁니다. `MOD_LOADER`도 `Auto`/빈 값이면 host selector가 결정합니다. Revise에서 Auto이면 기존 ZIP의 target을 보존하고, 사용자가 다른 target을 명시하면 migration으로 처리합니다.

`REFERENCE_MOD_URLS`에는 GitHub repository URL 또는 `owner/repo`를 공백·쉼표·세미콜론으로 여러 개 입력할 수 있습니다. 이 값은 검색 힌트가 아니라 사용자가 명시한 참고 후보이며, commit pin·permissive license·target compatibility·dependency closure·authoritative compile/proof를 모두 통과해야 실제 source reuse가 허용됩니다. 원본 repository는 수정하지 않습니다.

**Debug**는 PROMPT/모델 생성/모드 제작을 건너뛰고 `tools/full_project_audit.py`를 실행합니다. 개별 검사가 실패해도 다음 검사를 계속하며, 마지막에 전체 PASS/FAIL 목록과 상세 로그를 `mmm-debug-report.zip`으로 내려받습니다.

Full/Revise/Execute 실행 경로에는 source-only 우회 옵션이 없습니다. 제작 단계는 소스 생성 뒤 Gradle/GameTest/JAR 검증 단계로 계속 진행합니다.

원격 trajectory/temporary-skill 저장은 `ALLOW_REMOTE_TRAJECTORY_STORE=False`가 기본이며 사용자가 직접 `True`로 바꾼 실행에서만 허용됩니다.
